In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Testing FBCCA on benchmark data and comparing it to CCA results

In [9]:
from ssvep.datasets import load_benchmark
from ssvep import config

X, y, blocks = load_benchmark(1)

In [10]:
from ssvep.preprocessing import bandpass, notch, crop, filter_bank

X_filt = notch(X, fs=config.BENCHMARK_FS)

X_filt_cca = bandpass(X_filt, fs=config.BENCHMARK_FS, low=config.BANDPASS[0], high=config.BANDPASS[1])[:, None]
X_filt_fbcca = filter_bank(X_filt, fs=config.BENCHMARK_FS, n_subbands=config.N_SUBBANDS)

print(X_filt_cca.shape, X_filt_fbcca.shape)

start = config.BENCHMARK_ONSET + config.BENCHMARK_LATENCY

(30, 1, 9, 1500) (30, 5, 9, 1500)


In [11]:
from ssvep.cca import CCA

WINDOW_LENGTHS = [0.5, 1.0, 1.5, 2.0]

score_dict_cca = {}
score_dict_fbcca = {}

for window_length in WINDOW_LENGTHS:
    print(f"Window length: {window_length:.1f}s")

    X_window_cca = crop(X_filt_cca, fs=config.BENCHMARK_FS, start=start, duration=window_length)
    X_window_fbcca = crop(X_filt_fbcca, fs=config.BENCHMARK_FS, start=start, duration=window_length)

    cca = CCA(freqs=config.TARGET_FREQS, fs=config.BENCHMARK_FS)
    cca.fit(X_window_cca, y)
    score_cca = cca.score(X_window_cca, y)
    print(f"CCA score: {score_cca:.4f}")
    score_dict_cca[window_length] = score_cca

    fbcca = CCA(freqs=config.TARGET_FREQS, fs=config.BENCHMARK_FS)
    fbcca.fit(X_window_fbcca, y)
    score_fbcca = fbcca.score(X_window_fbcca, y)
    print(f"FBCCA score: {score_fbcca:.4f}")
    score_dict_fbcca[window_length] = score_fbcca


Window length: 0.5s
CCA score: 0.6000
FBCCA score: 0.7667
Window length: 1.0s
CCA score: 0.8333
FBCCA score: 0.9667
Window length: 1.5s
CCA score: 0.8333
FBCCA score: 0.9667
Window length: 2.0s
CCA score: 0.9333
FBCCA score: 1.0000
